# Módulo 1: People Analytics en Vertex AI — Notebook Práctico

## Objetivos
- Generar datos sintéticos realistas de RRHH
- Explorar las tipologías de datos en People Analytics
- Identificar riesgos: sesgos, re-identificación, proxy discrimination
- Conectar con el stack de GCP (BigQuery, Vertex AI)

**Nota:** Este notebook se puede ejecutar en Vertex AI Workbench o en cualquier entorno Python.

---
## 1. Setup e instalación de dependencias

In [ ]:
# Instalar dependencias (descomentar si es necesario)
# !pip install pandas numpy matplotlib seaborn scikit-learn faker

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Semilla para reproducibilidad
np.random.seed(42)

print("✅ Setup completado")

---
## 2. Generación de datos sintéticos de RRHH

Vamos a crear un dataset realista de una empresa ficticia de ~2.000 empleados.
Incluiremos datos estructurados, semiestructurados y derivados.

In [ ]:
n_empleados = 2000

# --- Datos estructurados ---
departamentos = ['Tecnología', 'Ventas', 'RRHH', 'Finanzas', 'Operaciones', 'Marketing', 'Legal', 'I+D']
niveles = ['Junior', 'Mid', 'Senior', 'Lead', 'Manager', 'Director']
ciudades = ['Madrid', 'Barcelona', 'Valencia', 'Bilbao', 'Sevilla', 'Remoto']
generos = ['Hombre', 'Mujer', 'No binario']

df = pd.DataFrame({
    'empleado_id': [f'EMP-{str(i).zfill(5)}' for i in range(1, n_empleados + 1)],
    'departamento': np.random.choice(departamentos, n_empleados, 
                                     p=[0.25, 0.20, 0.08, 0.10, 0.15, 0.10, 0.05, 0.07]),
    'nivel': np.random.choice(niveles, n_empleados, 
                              p=[0.25, 0.30, 0.20, 0.10, 0.10, 0.05]),
    'ciudad': np.random.choice(ciudades, n_empleados, 
                               p=[0.30, 0.25, 0.10, 0.10, 0.10, 0.15]),
    'genero': np.random.choice(generos, n_empleados, p=[0.52, 0.46, 0.02]),
    'edad': np.clip(np.random.normal(38, 8, n_empleados).astype(int), 22, 63),
    'antiguedad_meses': np.clip(np.random.exponential(36, n_empleados).astype(int), 1, 240),
})

# Salario base según nivel (con ruido)
salario_base = {'Junior': 28000, 'Mid': 38000, 'Senior': 50000, 
                'Lead': 60000, 'Manager': 70000, 'Director': 90000}
df['salario_bruto'] = df['nivel'].map(salario_base) + np.random.normal(0, 5000, n_empleados)
df['salario_bruto'] = df['salario_bruto'].clip(22000, 150000).round(0).astype(int)

# Meses desde última promoción
df['meses_sin_promocion'] = np.clip(
    (df['antiguedad_meses'] * np.random.uniform(0.3, 0.9, n_empleados)).astype(int), 
    0, 120
)

# Rating de desempeño (1-5) con sesgo de leniency
df['rating_desempeno'] = np.random.choice([1, 2, 3, 4, 5], n_empleados, 
                                           p=[0.02, 0.08, 0.25, 0.45, 0.20])

# Distancia al centro de trabajo (km)
df['distancia_km'] = np.where(
    df['ciudad'] == 'Remoto', 0,
    np.clip(np.random.exponential(15, n_empleados), 1, 80).round(1)
)

# Score de encuesta de clima (1-10)
df['score_clima'] = np.clip(np.random.normal(6.5, 1.5, n_empleados), 1, 10).round(1)

# Días de absentismo en último año
df['dias_absentismo'] = np.clip(np.random.exponential(5, n_empleados), 0, 60).astype(int)

# Horas de formación en último año
df['horas_formacion'] = np.clip(np.random.exponential(20, n_empleados), 0, 120).astype(int)

print(f"Dataset generado: {df.shape[0]} empleados, {df.shape[1]} variables")
df.head(10)

### 2.1 Introducir sesgos realistas (para detectarlos después)

En la realidad, los datos de RRHH contienen sesgos históricos. Vamos a simularlos intencionadamente para poder detectarlos en el análisis.

In [ ]:
# SESGO 1: Brecha salarial de género (~7% a favor de hombres, controlando por nivel)
mask_mujer = df['genero'] == 'Mujer'
df.loc[mask_mujer, 'salario_bruto'] = (df.loc[mask_mujer, 'salario_bruto'] * 0.93).astype(int)

# SESGO 2: Mujeres tienen más meses sin promoción (techo de cristal)
df.loc[mask_mujer, 'meses_sin_promocion'] = (
    df.loc[mask_mujer, 'meses_sin_promocion'] * 1.3
).astype(int)

# SESGO 3: Menos mujeres en niveles altos
# Reasignar: reducir probabilidad de Director/Lead para mujeres
reasignar = (df['genero'] == 'Mujer') & (df['nivel'].isin(['Director', 'Lead']))
n_reasignar = int(reasignar.sum() * 0.4)  # 40% de mujeres en niveles altos bajan
if n_reasignar > 0:
    idx_reasignar = df[reasignar].sample(n_reasignar, random_state=42).index
    df.loc[idx_reasignar, 'nivel'] = 'Senior'

# SESGO 4: Rating de desempeño ligeramente sesgado (lenguaje diferente pero mismo rating)
# Las mujeres reciben ratings iguales pero con menos varianza (menos 5s y menos 1s)
mask_mujer_extremo = mask_mujer & (df['rating_desempeno'] == 5)
n_ajustar_rating = int(mask_mujer_extremo.sum() * 0.25)
if n_ajustar_rating > 0:
    idx_rating = df[mask_mujer_extremo].sample(n_ajustar_rating, random_state=42).index
    df.loc[idx_rating, 'rating_desempeno'] = 4

print("✅ Sesgos realistas introducidos en el dataset")
print(f"  - Brecha salarial de género simulada")
print(f"  - Techo de cristal en promociones simulado")
print(f"  - Infrarrepresentación en niveles altos simulada")
print(f"  - Sesgo en ratings extremos simulado")

### 2.2 Generar variable objetivo: rotación voluntaria

Simulamos quién dejó la empresa en los últimos 12 meses. La probabilidad de irse depende de varios factores (incluyendo los sesgos introducidos).

In [ ]:
# Probabilidad base de rotación: ~15%
# Factores que aumentan probabilidad:
#   - Muchos meses sin promoción
#   - Score de clima bajo
#   - Salario bajo relativo al nivel
#   - Poca formación
#   - Alta distancia

# Normalizar features para el cálculo
from sklearn.preprocessing import MinMaxScaler

features_rotacion = ['meses_sin_promocion', 'score_clima', 'salario_bruto', 
                      'horas_formacion', 'distancia_km', 'dias_absentismo']

scaler = MinMaxScaler()
df_norm = pd.DataFrame(
    scaler.fit_transform(df[features_rotacion]),
    columns=features_rotacion
)

# Score de riesgo de rotación (combinación ponderada)
prob_rotacion = (
    0.30 * df_norm['meses_sin_promocion'] +       # Más tiempo sin promoción → más riesgo
    0.25 * (1 - df_norm['score_clima']) +           # Menos clima → más riesgo
    0.15 * (1 - df_norm['salario_bruto']) +         # Menos salario → más riesgo
    0.10 * (1 - df_norm['horas_formacion']) +       # Menos formación → más riesgo
    0.10 * df_norm['distancia_km'] +                # Más distancia → más riesgo
    0.10 * df_norm['dias_absentismo']               # Más absentismo → más riesgo
)

# Convertir a probabilidad y generar variable binaria
prob_rotacion = prob_rotacion.clip(0.02, 0.60)
df['rotacion'] = (np.random.random(n_empleados) < prob_rotacion).astype(int)

tasa_rotacion = df['rotacion'].mean() * 100
print(f"Tasa de rotación simulada: {tasa_rotacion:.1f}%")
print(f"Empleados que se fueron: {df['rotacion'].sum()}")
print(f"Empleados que se quedaron: {(1 - df['rotacion']).sum()}")

---
## 3. Exploración de datos: lo que un analista de People Analytics debe ver primero

Antes de cualquier modelo, necesitamos entender la distribución, calidad y posibles sesgos de los datos.

In [ ]:
# Vista general del dataset
print("=" * 60)
print("RESUMEN DEL DATASET")
print("=" * 60)
print(f"\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"\nVariables numéricas:")
print(df.describe().round(1).to_string())
print(f"\nVariables categóricas:")
for col in ['departamento', 'nivel', 'ciudad', 'genero']:
    print(f"\n  {col}:")
    print(f"    {df[col].value_counts().to_dict()}")
print(f"\nValores nulos: {df.isnull().sum().sum()}")

### 3.1 Distribución de la plantilla

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Distribución por departamento
df['departamento'].value_counts().plot(kind='barh', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Distribución por Departamento')
axes[0, 0].set_xlabel('Número de empleados')

# Distribución por nivel
nivel_order = ['Junior', 'Mid', 'Senior', 'Lead', 'Manager', 'Director']
df['nivel'].value_counts().reindex(nivel_order).plot(kind='barh', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Distribución por Nivel')
axes[0, 1].set_xlabel('Número de empleados')

# Distribución de edad
df['edad'].hist(bins=30, ax=axes[1, 0], color='mediumseagreen', edgecolor='white')
axes[1, 0].set_title('Distribución de Edad')
axes[1, 0].set_xlabel('Edad')

# Distribución de antigüedad
df['antiguedad_meses'].hist(bins=30, ax=axes[1, 1], color='mediumpurple', edgecolor='white')
axes[1, 1].set_title('Distribución de Antigüedad (meses)')
axes[1, 1].set_xlabel('Meses')

plt.suptitle('Panorama General de la Plantilla', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Detección de sesgos — Ejercicio práctico

Ahora vamos a buscar los sesgos que introdujimos (y quizás algunos que no esperábamos).

### 4.1 Brecha salarial de género

In [ ]:
# Análisis de brecha salarial bruta
brecha_bruta = df.groupby('genero')['salario_bruto'].mean()
print("BRECHA SALARIAL BRUTA (sin controlar por nivel)")
print("=" * 50)
for g in brecha_bruta.index:
    print(f"  {g}: €{brecha_bruta[g]:,.0f}")

brecha_pct = ((brecha_bruta['Hombre'] - brecha_bruta['Mujer']) / brecha_bruta['Hombre'] * 100)
print(f"\n  Brecha bruta: {brecha_pct:.1f}% a favor de hombres")

# Análisis controlado por nivel (brecha ajustada)
print("\n\nBRECHA SALARIAL AJUSTADA (controlando por nivel)")
print("=" * 50)
brecha_por_nivel = df.pivot_table(
    values='salario_bruto', 
    index='nivel', 
    columns='genero', 
    aggfunc='mean'
).round(0)

brecha_por_nivel['Brecha %'] = (
    (brecha_por_nivel['Hombre'] - brecha_por_nivel['Mujer']) / brecha_por_nivel['Hombre'] * 100
).round(1)

print(brecha_por_nivel.to_string())

In [ ]:
# Visualización de brecha salarial por nivel y género
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot salario por género y nivel
nivel_order = ['Junior', 'Mid', 'Senior', 'Lead', 'Manager', 'Director']
df_plot = df[df['genero'].isin(['Hombre', 'Mujer'])]

sns.boxplot(data=df_plot, x='nivel', y='salario_bruto', hue='genero',
            order=nivel_order, palette={'Hombre': 'steelblue', 'Mujer': 'coral'},
            ax=axes[0])
axes[0].set_title('Distribución Salarial por Nivel y Género')
axes[0].set_xlabel('Nivel')
axes[0].set_ylabel('Salario Bruto (€)')
axes[0].tick_params(axis='x', rotation=45)

# Proporción de género por nivel (techo de cristal)
gender_by_level = df[df['genero'].isin(['Hombre', 'Mujer'])].groupby(
    ['nivel', 'genero']).size().unstack(fill_value=0)
gender_pct = gender_by_level.div(gender_by_level.sum(axis=1), axis=0) * 100
gender_pct = gender_pct.reindex(nivel_order)

gender_pct.plot(kind='bar', stacked=True, ax=axes[1],
                color={'Hombre': 'steelblue', 'Mujer': 'coral'})
axes[1].set_title('Proporción de Género por Nivel (Techo de Cristal)')
axes[1].set_xlabel('Nivel')
axes[1].set_ylabel('Porcentaje (%)')
axes[1].axhline(y=50, color='gray', linestyle='--', alpha=0.5)
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Género')

plt.suptitle('Análisis de Equidad de Género', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Observa: la proporción de mujeres disminuye en niveles altos (techo de cristal simulado)")

### 4.2 Análisis de rotación por grupo

In [ ]:
# Tasa de rotación por diferentes cortes
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Por género
rot_genero = df.groupby('genero')['rotacion'].mean() * 100
rot_genero.plot(kind='bar', ax=axes[0, 0], color=['steelblue', 'coral', 'mediumseagreen'])
axes[0, 0].set_title('Tasa de Rotación por Género')
axes[0, 0].set_ylabel('Rotación (%)')
axes[0, 0].tick_params(axis='x', rotation=0)

# Por departamento
rot_depto = df.groupby('departamento')['rotacion'].mean().sort_values() * 100
rot_depto.plot(kind='barh', ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Tasa de Rotación por Departamento')
axes[0, 1].set_xlabel('Rotación (%)')

# Por nivel
rot_nivel = df.groupby('nivel')['rotacion'].mean().reindex(nivel_order) * 100
rot_nivel.plot(kind='bar', ax=axes[1, 0], color='coral')
axes[1, 0].set_title('Tasa de Rotación por Nivel')
axes[1, 0].set_ylabel('Rotación (%)')
axes[1, 0].tick_params(axis='x', rotation=45)

# Por rating de desempeño
rot_rating = df.groupby('rating_desempeno')['rotacion'].mean() * 100
rot_rating.plot(kind='bar', ax=axes[1, 1], color='mediumseagreen')
axes[1, 1].set_title('Tasa de Rotación por Rating de Desempeño')
axes[1, 1].set_ylabel('Rotación (%)')
axes[1, 1].set_xlabel('Rating')

plt.suptitle('Análisis de Rotación por Segmentos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Proxy discrimination — Detectar variables correlacionadas con categorías protegidas

In [ ]:
# ¿Qué variables son proxies de género?
# Codificar género como binario para correlación
df['genero_bin'] = (df['genero'] == 'Mujer').astype(int)

# Correlación punto-biserial de género con variables numéricas
from scipy import stats

print("ANÁLISIS DE PROXY DISCRIMINATION")
print("=" * 60)
print("\nCorrelación de cada variable con género (Mujer=1):")
print("-" * 60)

vars_numericas = ['salario_bruto', 'meses_sin_promocion', 'rating_desempeno',
                  'distancia_km', 'score_clima', 'dias_absentismo', 
                  'horas_formacion', 'antiguedad_meses', 'edad']

correlaciones = []
for var in vars_numericas:
    corr, p_value = stats.pointbiserialr(df['genero_bin'], df[var])
    significativo = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
    correlaciones.append({
        'Variable': var,
        'Correlación': round(corr, 3),
        'p-value': f"{p_value:.4f}",
        'Sig.': significativo,
        'Riesgo proxy': '⚠️ ALTO' if abs(corr) > 0.15 else '⚡ Medio' if abs(corr) > 0.05 else '✅ Bajo'
    })

df_corr = pd.DataFrame(correlaciones).sort_values('Correlación', key=abs, ascending=False)
print(df_corr.to_string(index=False))

print("\n💡 Variables con alta correlación con género son proxies potenciales.")
print("   Si las usas en un modelo, podrías estar discriminando indirectamente.")

# Limpiar columna auxiliar
df.drop('genero_bin', axis=1, inplace=True)

---
## 5. Riesgo de re-identificación — Ejercicio de k-anonimidad

Uno de los riesgos más subestimados en People Analytics. ¿Cuántos empleados son **únicos** al cruzar pocas variables?

In [ ]:
# k-anonimidad: ¿cuántos grupos tienen menos de k individuos?
# Combinamos quasi-identificadores: departamento, nivel, ciudad, género, rango de edad

df['rango_edad'] = pd.cut(df['edad'], bins=[20, 25, 30, 35, 40, 45, 50, 55, 60, 65], 
                          labels=['22-25', '26-30', '31-35', '36-40', '41-45', '46-50', '51-55', '56-60', '61-65'])

quasi_ids = ['departamento', 'nivel', 'ciudad', 'genero', 'rango_edad']

# Contar tamaño de cada grupo
grupos = df.groupby(quasi_ids).size().reset_index(name='n_personas')

print("ANÁLISIS DE K-ANONIMIDAD")
print("=" * 60)
print(f"\nQuasi-identificadores: {', '.join(quasi_ids)}")
print(f"Total de combinaciones únicas: {len(grupos)}")
print(f"\nDistribución del tamaño de grupos:")
print(f"  Grupos con 1 persona (re-identificables): {(grupos['n_personas'] == 1).sum()} ({(grupos['n_personas'] == 1).mean()*100:.1f}%)")
print(f"  Grupos con 2 personas: {(grupos['n_personas'] == 2).sum()}")
print(f"  Grupos con 3-5 personas: {((grupos['n_personas'] >= 3) & (grupos['n_personas'] <= 5)).sum()}")
print(f"  Grupos con >5 personas: {(grupos['n_personas'] > 5).sum()}")

# Personas en riesgo
personas_riesgo = grupos[grupos['n_personas'] <= 2]['n_personas'].sum()
print(f"\n⚠️ Personas en grupos de ≤2 (alto riesgo de re-identificación): {personas_riesgo} ({personas_riesgo/n_empleados*100:.1f}%)")

# Visualización
fig, ax = plt.subplots(figsize=(10, 5))
bins = [0, 1, 2, 3, 5, 10, 20, 50, 100]
grupos['n_personas'].clip(upper=100).hist(bins=bins, ax=ax, color='coral', edgecolor='white')
ax.set_title('Distribución del tamaño de grupos (k-anonimidad)', fontsize=14)
ax.set_xlabel('Número de personas en el grupo')
ax.set_ylabel('Número de grupos')
ax.axvline(x=5, color='red', linestyle='--', label='Umbral k=5')
ax.legend()
plt.tight_layout()
plt.show()

print("\n💡 Regla práctica: nunca publicar estadísticas de grupos con menos de 5 personas.")
print("   En Vertex AI: asegurarse de que los datos de entrenamiento estén agregados adecuadamente.")

df.drop('rango_edad', axis=1, inplace=True)

---
## 6. Feature importance y su relación con sesgos

Entrenamos un modelo simple de predicción de rotación para ver qué features importan. Si `genero` o sus proxies tienen alta importancia, tenemos un problema.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Preparar features
features = ['edad', 'antiguedad_meses', 'salario_bruto', 'meses_sin_promocion',
            'rating_desempeno', 'distancia_km', 'score_clima', 'dias_absentismo',
            'horas_formacion']

# Añadir variables categóricas codificadas
le_dept = LabelEncoder()
le_nivel = LabelEncoder()
le_genero = LabelEncoder()

df['dept_encoded'] = le_dept.fit_transform(df['departamento'])
df['nivel_encoded'] = le_nivel.fit_transform(df['nivel'])
df['genero_encoded'] = le_genero.fit_transform(df['genero'])

features_all = features + ['dept_encoded', 'nivel_encoded', 'genero_encoded']

X = df[features_all]
y = df['rotacion']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Entrenar Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

# Evaluación
y_pred = rf.predict(X_test)
print("MODELO DE PREDICCIÓN DE ROTACIÓN")
print("=" * 60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Se queda', 'Se va']))

# Feature importance
importances = pd.DataFrame({
    'Feature': features_all,
    'Importancia': rf.feature_importances_
}).sort_values('Importancia', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['red' if f == 'genero_encoded' else 'orange' if f in ['salario_bruto', 'meses_sin_promocion'] else 'steelblue' 
          for f in importances['Feature']]
importances.plot(kind='barh', x='Feature', y='Importancia', ax=ax, color=colors, legend=False)
ax.set_title('Feature Importance — Predicción de Rotación', fontsize=14)
ax.set_xlabel('Importancia')

# Leyenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label='⚠️ Categoría protegida'),
    Patch(facecolor='orange', label='⚡ Posible proxy de género'),
    Patch(facecolor='steelblue', label='✅ Feature neutral')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

print("\n💡 Si 'genero_encoded' tiene alta importancia → el modelo está usando género para predecir.")
print("   Si 'salario_bruto' o 'meses_sin_promocion' son top features → pueden ser proxies de género.")
print("   Esto es exactamente lo que se discute en el Art. 22 GDPR y en fairness ML.")

### 6.1 Fairness check: ¿El modelo predice diferente por género?

In [ ]:
# Análisis de equidad del modelo
df_test = X_test.copy()
df_test['y_real'] = y_test.values
df_test['y_pred'] = y_pred
df_test['genero'] = le_genero.inverse_transform(df_test['genero_encoded'])

print("ANÁLISIS DE EQUIDAD DEL MODELO")
print("=" * 60)

for genero in ['Hombre', 'Mujer']:
    mask = df_test['genero'] == genero
    subset = df_test[mask]
    
    tp = ((subset['y_pred'] == 1) & (subset['y_real'] == 1)).sum()
    fp = ((subset['y_pred'] == 1) & (subset['y_real'] == 0)).sum()
    fn = ((subset['y_pred'] == 0) & (subset['y_real'] == 1)).sum()
    tn = ((subset['y_pred'] == 0) & (subset['y_real'] == 0)).sum()
    
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  # True Positive Rate (Recall)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate
    pred_positive_rate = (subset['y_pred'] == 1).mean()  # Demographic parity
    
    print(f"\n  {genero} (n={len(subset)}):")
    print(f"    True Positive Rate (Recall):  {tpr:.3f}")
    print(f"    False Positive Rate:          {fpr:.3f}")
    print(f"    Predicted Positive Rate:      {pred_positive_rate:.3f}")
    print(f"    Actual Positive Rate:         {subset['y_real'].mean():.3f}")

print("\n" + "-" * 60)
print("\nMétricas de equidad:")

hombre_pred_rate = df_test[df_test['genero'] == 'Hombre']['y_pred'].mean()
mujer_pred_rate = df_test[df_test['genero'] == 'Mujer']['y_pred'].mean()
dp_ratio = min(hombre_pred_rate, mujer_pred_rate) / max(hombre_pred_rate, mujer_pred_rate)

print(f"  Demographic Parity Ratio: {dp_ratio:.3f} (ideal = 1.0, aceptable > 0.8)")
print(f"  {'✅ Dentro del umbral aceptable' if dp_ratio > 0.8 else '⚠️ FUERA del umbral — revisar sesgo'}")

print("\n💡 En Vertex AI, estas métricas se calculan automáticamente en Model Evaluation.")
print("   Se pueden configurar slices por cualquier atributo sensible.")

---
## 7. Conexión con GCP: Cómo llevar esto a Vertex AI

Este notebook se ha ejecutado localmente, pero en un entorno real usaríamos el stack completo de GCP.

In [ ]:
# Este bloque es INFORMATIVO — muestra cómo sería el código en Vertex AI
# No ejecutar a menos que estés en un entorno GCP configurado

codigo_vertex = """
# ============================================================
# CÓDIGO DE REFERENCIA: Cómo hacer esto en Vertex AI + BigQuery
# ============================================================

# 1. Subir datos a BigQuery
from google.cloud import bigquery

client = bigquery.Client(project='mi-proyecto-people-analytics')
table_id = 'mi-proyecto.people_analytics.empleados'

job = client.load_table_from_dataframe(df, table_id)
job.result()  # Esperar a que termine
print(f"Datos cargados en {table_id}")

# 2. Entrenar modelo con AutoML Tabular en Vertex AI
from google.cloud import aiplatform

aiplatform.init(
    project='mi-proyecto-people-analytics',
    location='europe-west1',  # Madrid region
)

dataset = aiplatform.TabularDataset.create(
    display_name='people-analytics-rotacion',
    bq_source=f'bq://{table_id}'
)

job = aiplatform.AutoMLTabularTrainingJob(
    display_name='rotacion-prediction-v1',
    optimization_prediction_type='classification',
    column_transformations=[
        {"numeric": {"column_name": "edad"}},
        {"numeric": {"column_name": "antiguedad_meses"}},
        {"numeric": {"column_name": "salario_bruto"}},
        {"numeric": {"column_name": "meses_sin_promocion"}},
        {"numeric": {"column_name": "rating_desempeno"}},
        {"numeric": {"column_name": "score_clima"}},
        {"categorical": {"column_name": "departamento"}},
        {"categorical": {"column_name": "nivel"}},
        # NOTA: NO incluimos 'genero' como feature del modelo
    ],
)

model = job.run(
    dataset=dataset,
    target_column='rotacion',
    training_fraction_split=0.8,
    validation_fraction_split=0.1,
    test_fraction_split=0.1,
    budget_milli_node_hours=1000,  # ~1 hora de entrenamiento
)

# 3. Evaluar con fairness metrics
# En la consola de Vertex AI > Model Registry > Evaluation
# Configurar slices por 'genero' para ver métricas de equidad

# 4. Feature Attributions (Explainability)
explanations = model.explain(instances=[sample_instance])
# Revisar que no haya proxies con alta atribución

# 5. Model Monitoring (post-deploy)
endpoint = model.deploy(
    deployed_model_display_name='rotacion-v1',
    machine_type='n1-standard-4',
)

# Configurar alertas de drift y sesgo
from google.cloud.aiplatform import model_monitoring

monitoring_job = aiplatform.ModelDeploymentMonitoringJob.create(
    display_name='rotacion-monitoring',
    endpoint=endpoint,
    logging_sampling_strategy=model_monitoring.RandomSampleConfig(sample_rate=0.5),
    schedule_config=model_monitoring.ScheduleConfig(monitor_interval=3600),
    # Detectar drift en features y predicciones
)
"""

print(codigo_vertex)
print("\n" + "=" * 60)
print("Este código se ejecutará en el Módulo 2 cuando tengamos")
print("acceso al entorno de GCP configurado.")
print("=" * 60)

---
## 8. Resumen y siguientes pasos

### Lo que hemos hecho en este notebook:
1. **Generado datos sintéticos** realistas de una empresa de 2.000 empleados
2. **Introducido sesgos** controlados (brecha salarial, techo de cristal, sesgo en ratings)
3. **Explorado** la distribución de la plantilla
4. **Detectado sesgos** con análisis estadístico y visual
5. **Analizado proxy discrimination** (correlaciones con categorías protegidas)
6. **Evaluado k-anonimidad** (riesgo de re-identificación)
7. **Entrenado un modelo** simple y analizado su fairness
8. **Conectado con Vertex AI** (código de referencia para el Módulo 2)

### Ideas clave:
- Los datos de RRHH **contienen sesgos históricos** que los modelos heredan
- **Eliminar una variable** (ej: género) no elimina el sesgo si hay proxies
- La **k-anonimidad** es fundamental antes de compartir cualquier dato
- Las **métricas de fairness** deben ser parte del proceso de evaluación
- **Vertex AI** ofrece herramientas nativas para abordar estos retos

### En el Módulo 2:
- Configuraremos el entorno real en GCP
- Subiremos estos datos a BigQuery
- Entrenaremos el modelo en Vertex AI AutoML
- Usaremos Model Evaluation con slices de fairness
- Desplegaremos con monitoring

In [ ]:
# Guardar el dataset para uso en módulos posteriores
df.to_csv('people_analytics_sintetico.csv', index=False)
print(f"✅ Dataset guardado: people_analytics_sintetico.csv ({df.shape[0]} filas, {df.shape[1]} columnas)")
print("\n📋 Columnas disponibles:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")